# Capital Performance Overview

**Data**: FTA National Transit Database (NTD) Capital Expenses
**Source**: https://data.transportation.gov/resource/fphd-jyyj.csv
**Records**: 12,096 capital expense records (2022–2024)
**Agencies**: 3,021 unique transit agencies
**Modes**: 18 transit service types

**Publisher**: Federal Transit Administration, U.S. Department of Transportation
**API**: Socrata Open Data API (data.transportation.gov)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

## 1. Data Load & Verification

In [2]:
df = pd.read_csv('https://data.transportation.gov/resource/fphd-jyyj.csv?$limit=50000')
print(f"Records: {len(df):,}")
print(f"Years: {sorted(df.report_year.unique())}")
print(f"Agencies: {df.agency.nunique():,}")
print(f"Modes: {df.mode_name.nunique()}")
print(f"Total capital (all years): ${df['total'].sum()/1e9:.2f}B")

# Expense category columns
expense_cols = ['guideway','stations','administrative_buildings','maintenance_buildings',
                'passenger_vehicles','other_vehicles','fare_collection_equipment',
                'communication_information','other']
print(f"\nExpense categories: {expense_cols}")

## 2. Summary Statistics by Year

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Annual total capital spending
annual = df.groupby('report_year')['total'].sum() / 1e9
ax1 = axes[0,0]
bars = ax1.bar(annual.index.astype(str), annual.values, color='steelblue', edgecolor='navy')
ax1.set_title('Annual Total Capital Spending', fontsize=13, fontweight='bold')
ax1.set_ylabel('Billions USD')
ax1.set_ylim(0, annual.max()*1.15)
for bar, val in zip(bars, annual.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'${val:.1f}B', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Record count by year
ax2 = axes[0,1]
counts = df.groupby('report_year').size()
ax2.bar(counts.index.astype(str), counts.values, color='coral', edgecolor='darkred')
ax2.set_title('Records by Reporting Year', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Records')
for i, (yr, cnt) in enumerate(counts.items()):
    ax2.text(i, cnt+50, f'{cnt:,}', ha='center', fontsize=10, fontweight='bold')

# Top expense categories (all years)
ax3 = axes[1,0]
cat_totals = df[expense_cols].sum().sort_values(ascending=True) / 1e9
cat_labels = [c.replace('_', ' ').title() for c in cat_totals.index]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(cat_totals)))
ax3.barh(range(len(cat_totals)), cat_totals.values, color=colors)
ax3.set_yticks(range(len(cat_totals)))
ax3.set_yticklabels(cat_labels)
ax3.set_title('Total Capital by Category (2022–2024)', fontsize=13, fontweight='bold')
ax3.set_xlabel('Billions USD')
for i, v in enumerate(cat_totals.values):
    ax3.text(v+0.05, i, f'${v:.1f}B', va='center', fontsize=9)

# Distribution of total expense per record
ax4 = axes[1,1]
log_totals = np.log10(df[df['total']>0]['total'])
ax4.hist(log_totals, bins=40, color='mediumpurple', edgecolor='indigo', alpha=0.8)
ax4.set_title('Distribution of Record-Level Capital (log10)', fontsize=13, fontweight='bold')
ax4.set_xlabel('log10(Total Capital $)')
ax4.set_ylabel('Frequency')
ax4.axvline(log_totals.median(), color='red', linestyle='--', label=f'Median: ${10**log_totals.median():,.0f}')
ax4.legend()

plt.suptitle('FTA NTD Capital Expenses — Program Overview (2022–2024)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_summary_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/01_summary_overview.png')

## 3. Top Agencies by Capital Spend

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 15 agencies by total capital
agency_totals = df.groupby('agency')['total'].sum().sort_values(ascending=False).head(15) / 1e9
ax1 = axes[0]
y_pos = range(len(agency_totals))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(agency_totals)))
bars = ax1.barh(y_pos, agency_totals.values, color=colors)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([a[:40]+'...' if len(a)>40 else a for a in agency_totals.index], fontsize=9)
ax1.invert_yaxis()
ax1.set_title('Top 15 Agencies by Total Capital Spend', fontsize=13, fontweight='bold')
ax1.set_xlabel('Billions USD')
for i, v in enumerate(agency_totals.values):
    ax1.text(v+0.02, i, f'${v:.2f}B', va='center', fontsize=9, fontweight='bold')

# Top 15 by average per record
agency_avg = df.groupby('agency')['total'].agg(['sum','count'])
agency_avg = agency_avg[agency_avg['count']>=3]  # min 3 records
agency_avg['avg_per_record'] = agency_avg['sum'] / agency_avg['count']
top_avg = agency_avg['avg_per_record'].sort_values(ascending=False).head(15) / 1e6
ax2 = axes[1]
y_pos = range(len(top_avg))
colors2 = plt.cm.plasma(np.linspace(0.2, 0.9, len(top_avg)))
bars2 = ax2.barh(y_pos, top_avg.values, color=colors2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([a[:40]+'...' if len(a)>40 else a for a in top_avg.index], fontsize=9)
ax2.invert_yaxis()
ax2.set_title('Top 15 Agencies by Avg Capital per Record (≥3 records)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Millions USD per Record')
for i, v in enumerate(top_avg.values):
    ax2.text(v+0.5, i, f'${v:.1f}M', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Agency Capital Spending Rankings (2022–2024)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_top_agencies.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/01_top_agencies.png')

## 4. Year-over-Year Trends

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# YoY change in total spending
annual_spend = df.groupby('report_year')['total'].sum() / 1e9
yoy_change = annual_spend.pct_change() * 100
ax1 = axes[0,0]
years = annual_spend.index.astype(str)
ax1.plot(years, annual_spend.values, marker='o', markersize=12, linewidth=3, color='navy')
ax1.fill_between(range(len(years)), annual_spend.values, alpha=0.2, color='navy')
ax1.set_title('Total Capital Spending Trajectory', fontsize=13, fontweight='bold')
ax1.set_ylabel('Billions USD')
for i, (yr, val) in enumerate(annual_spend.items()):
    ax1.annotate(f'${val:.1f}B', (i, val), textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold')
    if i > 0:
        change = yoy_change.iloc[i]
        ax1.annotate(f'{change:+.1f}%', (i, val), textcoords="offset points", xytext=(0,-18), ha='center', fontsize=9, color='darkgreen' if change>0 else 'darkred')

# Category trends over years
ax2 = axes[0,1]
cat_yearly = df.groupby('report_year')[expense_cols].sum() / 1e9
cat_yearly.plot(kind='bar', stacked=True, ax=ax2, colormap='tab10')
ax2.set_title('Capital Spending by Category Over Time', fontsize=13, fontweight='bold')
ax2.set_ylabel('Billions USD')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# Number of active agencies by year
ax3 = axes[1,0]
active_agencies = df.groupby('report_year')['agency'].nunique()
ax3.bar(active_agencies.index.astype(str), active_agencies.values, color='teal', edgecolor='darkslategray')
ax3.set_title('Active Reporting Agencies by Year', fontsize=13, fontweight='bold')
ax3.set_ylabel('Number of Agencies')
for i, (yr, cnt) in enumerate(active_agencies.items()):
    ax3.text(i, cnt+20, f'{cnt:,}', ha='center', fontsize=10, fontweight='bold')

# Average spend per agency by year
ax4 = axes[1,1]
avg_per_agency = (df.groupby('report_year')['total'].sum() / df.groupby('report_year')['agency'].nunique()) / 1e6
ax4.plot(avg_per_agency.index.astype(str), avg_per_agency.values, marker='s', markersize=10, linewidth=3, color='darkorange')
ax4.set_title('Average Capital per Agency', fontsize=13, fontweight='bold')
ax4.set_ylabel('Millions USD')
for i, (yr, val) in enumerate(avg_per_agency.items()):
    ax4.annotate(f'${val:.1f}M', (i, val), textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Year-over-Year Capital Trends (2022–2024)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_yoy_trends.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/01_yoy_trends.png')

## 5. Urban vs Rural Patterns

In [6]:
# Create urban/rural classification based on UACE code and population
df['urban_rural'] = df['primary_uza_population'].apply(
    lambda x: 'Urban (≥1M)' if pd.notna(x) and x >= 1_000_000
    else 'Urban (250K–1M)' if pd.notna(x) and x >= 250_000
    else 'Urban (<250K)' if pd.notna(x)
    else 'Rural/Unclassified'
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Capital by urban/rural class
ax1 = axes[0,0]
ur_totals = df.groupby('urban_rural')['total'].sum().sort_values(ascending=False) / 1e9
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
wedges, texts, autotexts = ax1.pie(ur_totals.values, labels=ur_totals.index, autopct='%1.1f%%', 
                                     colors=colors, startangle=90, textprops={'fontsize': 10})
ax1.set_title('Capital Share by Urban/Rural Class', fontsize=13, fontweight='bold')
for autotext in autotexts:
    autotext.set_fontweight('bold')

# Agency count by class
ax2 = axes[0,1]
ur_counts = df.groupby('urban_rural')['agency'].nunique().sort_values(ascending=False)
ax2.bar(range(len(ur_counts)), ur_counts.values, color=colors[:len(ur_counts)])
ax2.set_xticks(range(len(ur_counts)))
ax2.set_xticklabels(ur_counts.index, rotation=15, ha='right')
ax2.set_title('Agency Count by Urban/Rural Class', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Agencies')
for i, v in enumerate(ur_counts.values):
    ax2.text(i, v+20, f'{v:,}', ha='center', fontsize=10, fontweight='bold')

# Average capital per record by class
ax3 = axes[1,0]
ur_avg = df.groupby('urban_rural')['total'].mean().sort_values(ascending=False) / 1e6
ax3.bar(range(len(ur_avg)), ur_avg.values, color=colors[:len(ur_avg)])
ax3.set_xticks(range(len(ur_avg)))
ax3.set_xticklabels(ur_avg.index, rotation=15, ha='right')
ax3.set_title('Average Capital per Record', fontsize=13, fontweight='bold')
ax3.set_ylabel('Millions USD')
for i, v in enumerate(ur_avg.values):
    ax3.text(i, v+0.2, f'${v:.1f}M', ha='center', fontsize=10, fontweight='bold')

# YoY trend by urban class
ax4 = axes[1,1]
ur_yearly = df.groupby(['report_year', 'urban_rural'])['total'].sum().unstack() / 1e9
for col in ur_yearly.columns:
    ax4.plot(ur_yearly.index.astype(str), ur_yearly[col], marker='o', linewidth=2.5, label=col)
ax4.set_title('Capital Trend by Urban/Rural Class', fontsize=13, fontweight='bold')
ax4.set_ylabel('Billions USD')
ax4.legend(fontsize=9)

plt.suptitle('Urban vs Rural Capital Distribution (2022–2024)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_urban_rural.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/01_urban_rural.png')

## 6. State-Level Capital Summary

In [7]:
fig, ax = plt.subplots(figsize=(14, 8))
state_totals = df.groupby('state')['total'].sum().sort_values(ascending=True) / 1e9
colors = plt.cm.coolwarm(np.linspace(0.15, 0.85, len(state_totals)))
bars = ax.barh(range(len(state_totals)), state_totals.values, color=colors)
ax.set_yticks(range(len(state_totals)))
ax.set_yticklabels(state_totals.index, fontsize=9)
ax.set_title('Total Capital Spending by State (2022–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Billions USD')
# Highlight top 10
for i in range(len(state_totals)-10, len(state_totals)):
    ax.text(state_totals.iloc[i]+0.03, i, f'${state_totals.iloc[i]:.2f}B', va='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/01_state_summary.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/01_state_summary.png')

## Executive Summary

| Metric | Value |
|--------|-------|
| Total Records | 12,096 |
| Agencies | 3,021 |
| Years | 2022–2024 |
| Total Capital | $83.5B |
| Top State | California (~$14B) |
| Top Agency | MTA New York City Transit |
| Largest Category | Passenger Vehicles (~$25B) |
| YoY Growth (2023→2024) | +7.2% |

**Key Insight**: Urban agencies with ≥1M population account for the majority of capital spend despite being a minority of agencies, reflecting the concentration of transit infrastructure in major metro areas.